![image info](https://user-images.githubusercontent.com/91945811/146929832-e78ff1f4-2739-41b1-acd3-019a39b6a42c.png)

# **Bericht zur Verkaufsdatenanalyse 2019**

## Einleitung – Ziele der Analyse
Die vorliegende Analyse hat das Ziel, die Verkaufsdaten des Jahres 2019 umfassend zu untersuchen. Ziel ist es, wichtige Erkenntnisse zu gewinnen, darunter:
- Identifikation umsatzstarker Produkte und Städte
- Erkennung saisonaler Schwankungen und Verkaufstrends
- Untersuchung des Kaufverhaltens der Kunden
- Analyse von Produktkombinationen
- Identifikation von Anomalien in den Verkaufsdaten
- Ableitung von Handlungsempfehlungen zur Optimierung des Verkaufs

### **Datenverständnis und erste Explorationsanalyse**
- Lade die Datei **exported_sales_2019.csv** in ein Pandas-DataFrame und verschaffe dir einen ersten Überblick.
- Untersuche die Struktur der Daten:
  - Welche Spalten sind enthalten?
  - Welche Datentypen haben die Spalten?
  - Gibt es fehlende Werte oder Inkonsistenzen in den Daten?

In [ ]:
# Bibliotheken Laden

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

# Daten einlesen
df = pd.read_csv("exported_sales_2019.csv")

In [ ]:
# erste Übersicht der Spalten
df.info()

In [ ]:
# df anzeigen
display(df)

Die Datei enthält **X Zeilen** und **y Spalten**:

Hier soll eine kurze Erläuterung der Spalten stehen

#### **Aufgaben**
1. **Fehlende Werte prüfen**  
   - prüfe, ob es Zeilen mit fehlenden Werten gibt und entferne diese wenn nötig.

2. **Datentypen anpassen**  
   - `Quantity Ordered` und `Price Each` in **numerische Werte** umwandeln.
   - `Order Date` in ein **DateTime-Format** umwandeln.

3. **Transformation der Daten**
   - Extrahiere aus der **Purchase Address** die Stadt und speichere sie in einer neuen Spalte **City**.(Tipp: verwende `split(', ')`)
   - Berechne **Total Sales** (Gesamtumsatz pro Bestellung) und **Month** (Monat der Bestellung). 

4. **Duplikate entfernen**
   - Es gibt **11.893 Duplikate** im Datensatz, die du entfernen musst.

5. **Index zurücksetzten**
   - Damit der Index nach dem Entfernen einiger Einträge wieder passt, musst du den Index noch zurücksetzte.
  

In [ ]:
# Fehlende Werte prüfen
df.isnull().sum()

In [ ]:
# Duplikate entfernen
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates()
df = df.reset_index(drop=True)

In [ ]:
# datentypen anpassen
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%y %H:%M")

In [ ]:
# Neue Spalten bilden
df["City"] = df["Purchase Address"].str.split(", ").str[1]
df["Total Sales"] = df["Quantity Ordered"] * df["Price Each"]
df["Month"] = df["Order Date"].dt.month

In [ ]:
# df erneut anzeigen:
display(df)

### **Ergebnisse der Datenbereinigung & Transformation**
- **fehlende Werte** wurden überprüft.
- Die Spalten **Quantity Ordered** und **Price Each** wurden in numerische Werte umgewandelt.
- Die Spalte **Order Date** wurde in ein **DateTime-Format** konvertiert.
- Die **City** wurde aus der **Purchase Address** extrahiert.
- Der **Total Sales** (Gesamtumsatz pro Bestellung) wurde berechnet.
- Der **Month** (Monat der Bestellung) wurde extrahiert.
- Alle **Duplikate** wurden entfernt.

Nach Bereinigung der Daten stehen 185.652 vollständige Bestellungen zur Verfügung.

Die Daten sind nun bereit für die Analyse.

---

## **Deskriptive Statistik und Kennzahlenberechnung**
- Berechne folgende Verkaufskennzahlen:
  - **Gesamtumsatz für das Jahr 2019**
  - **Durchschnittlicher Bestellwert**
  - **Anzahl der Bestellungen pro Monat**
  - **Beliebteste Produktkategorien oder Artikel (nach Anzahl der Verkäufe oder Umsatz)**
  - **Kunden mit dem höchsten Bestellvolumen** (lass dir für die Kunden jeweils die `Purchase Address` ausgeben.)

In [ ]:
# erste deskriptive statistik mit describe():
df.describe()

In [ ]:
# Gesamtumsatz für das Jahr 2019
total_sales_2019 = df["Total Sales"].sum()


# Durchschnittlicher Bestellwert
average_order_value = df["Total Sales"].mean()

# Anzahl der Bestellungen pro Monat
monthly_sales = df.groupby(df["Order Date"].dt.to_period("M"))["Total Sales"].count()

# Beliebteste Produktkategorien oder Artikel (nach Anzahl der Verkäufe und Umsatz)
sales_by_quantity = df.groupby("Product")["Quantity Ordered"].sum().sort_values(ascending=False)
best_sellers_by_quantity = sales_by_quantity.head(10)

sales_by_revenue = df.groupby("Product")["Total Sales"].sum().sort_values(ascending=False)
best_sellers_by_revenue = sales_by_revenue.head(10)

# Kunden mit dem höchsten Bestellvolumen
sales_by_customer = df.groupby("Purchase Address")["Total Sales"].sum().sort_values(ascending=False)
top_customer = sales_by_customer.head(10)

In [ ]:
# Ergebnisse anzeigen
print(f"Der Gesamtumsatz für das Jahr 2019 ist {total_sales_2019}.")
print(f"Der durchschnittliche Bestellwert ist {average_order_value}.")

In [ ]:
print("\nBestellungen pro Monat:")
print(monthly_sales)
print(f"\nStärkster Monat: {monthly_sales.idxmax()} mit {monthly_sales.max():,} Bestellungen")
print(f"Schwächster Monat: {monthly_sales.idxmin()} mit {monthly_sales.min():,} Bestellungen")

In [ ]:
print("\nTop 10 Produkte nach Verkaufszahlen:")
print(best_sellers_by_quantity)
print(f"Gesamte verkaufte Einheiten der Top 10: {best_sellers_by_quantity.sum():,}")

In [ ]:
print("Top 10 Produkte nach Umsatz:")
print(best_sellers_by_revenue)
print(f"\nGesamtumsatz der Top 10: {best_sellers_by_revenue.sum():,.2f}")

In [ ]:
print("\nTop 10 Kunden nach Bestellvolumen:")
print(top_customer)
print(f"\nHöchstes Bestellvolumen: {top_customer.iloc[0]:,.2f}")

### **Ergebnisse der deskriptiven Statistik & Kennzahlenberechnung**
1. **Gesamtumsatz für das Jahr 2019:** **34.47 Millionen USD**
2. **Durchschnittlicher Bestellwert:** **185.6 USD**
3. **Anzahl der Bestellungen pro Monat:**
   - Höchste Anzahl an Bestellungen: **Dezember (24.944 Bestellungen)**
   - Niedrigste Anzahl an Bestellungen: **Januar (9.665 Bestellungen)**
4. **Top 10 meistverkaufte Produkte (nach Anzahl der Verkäufe):**
   - **AAA Batteries (4-pack):** 30.981 Einheiten
   - **AA Batteries (4-pack):** 27.615 Einheiten
   - **USB-C Charging Cable:** 23.927 Einheiten
   - **Lightning Charging Cable:** 23.163 Einheiten
   - **Wired Headphones:** 20.520 Einheiten
   - **Apple Airpods Headphones:** 15.633 Einheiten
   - **Bose SoundSport Headphones:** 13.427 Einheiten
   - **27in FHD Monitor:** 7.538 Einheiten
   - **iPhone:** 6.845 Einheiten
   - **27in 4K Gaming Monitor:** 6.238 Einheiten

5. **Top 10 umsatzstärkste Produkte (nach Umsatz):**
   - **Macbook Pro Laptop:** **8.03 Mio. USD**
   - **iPhone:** **4.79 Mio. USD**
   - **ThinkPad Laptop:** **4.13 Mio. USD**
   - **Google Phone:** **3.32 Mio. USD**
   - **27in 4K Gaming Monitor:** **2.43 Mio. USD**
   - **34in Ultrawide Monitor:** **2.35 Mio. USD**
   - **Apple Airpods Headphones:** **2.34 Mio. USD**
   - **Flatscreen TV:** **1.44 Mio. USD**
   - **Bose SoundSport Headphones:** **1.34 Mio. USD**
   - **27in FHD Monitor:** **1.13 Mio. USD**

6. **Kunden mit dem höchsten Bestellvolumen:**
   - **Höchste Bestellsumme:** **4.379,99 USD** (San Francisco, CA)
   - **Zweitgrößte Bestellsumme:** **4.100,00 USD** (Seattle, WA sowie Atlanta, GA)
   - Weitere Städte mit hoher Bestellsumme: **New York City, San Francisco, Los Angeles**

---

## **Datenvisualisierung**

Erstelle verschiedene Diagramme zur Visualisierung der Verkaufsdaten
- Liniendiagramm: Anzahl Bestellungen pro Monat
- Balkendiagramm: Umsatz pro Produktkategorie, sowie die beliebtesten Produkte (Anzahl Verkäufe)
- Kreisdiagramm: Verteilung der Verkäufe auf verschiedene Regionen
- Histogramm: Verteilung der Bestellwerte
- Balkendiagamm: Wochentag vs. Umsatz (Gibt es umsatzstarke Tage?)
- Balkendiagramm: Verteilung der Bestellungen pro Stunde

Welche Muster kannst du aus den Diagrammen ableiten?

In [ ]:
# Erstellen der Visualisierungen
# Umsatzentwicklung pro Monat (Liniendiagramm)
plt.figure(figsize=(12, 6))
plt.plot(
    monthly_sales.index.astype(str),
    monthly_sales.values,
    marker="o",
    linewidth=2,
    markersize=6,
    color="#2E8B57",
)
plt.title("Anzahl Bestellungen pro Monat 2019", fontsize=16, fontweight="bold")
plt.xlabel("Monat", fontsize=12)
plt.ylabel("Anzahl Bestellungen", fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.ticklabel_format(style="plain", axis="y")

for i, v in enumerate(monthly_sales.values):
    plt.annotate(f"{v:,}", (i, v), textcoords="offset points", xytext=(0, 10), ha="center")

plt.tight_layout()
plt.show()

In [ ]:
# Umsatz pro Produktkategorie (Balkendiagramm)


In [ ]:
# Beliebteste Produkte (Balkendiagramm)

In [ ]:
# Verteilung der Verkäufe nach Städten (Kreisdiagramm)


In [ ]:
# Histogramm: Verteilung der Bestellwerte


In [ ]:
# Balkendiagramm: Umsatz nach Wochentag


In [ ]:
# Verteilung pro Stunde


### **Ergebnisse der Datenvisualisierung**

An dieser Stelle die Ergebnisse kurz zusammenfassen.

# Zeitreihenanalyse

Benutze nun die Funktion `seasonal_decompose` aus der Bibliothek `statsmodels.tsa.seasonal`, um eine Zeitreihenanalyse durchzuführen.

- Berechne hierfür zunächst die täglichen Umsätze. Achte darauf, dass `Order Date` auch die Stunden beinhaltet; begrenze dies also zunächst durch `dt.date` nur auf die Tage
- führe eine Saisonale Dekomposition durch. Da unsere Daten nur ca. ein Jahr umfassen, kannst du die Periodizität auf ca. einen Monat, also 30 Tage festlegen.
- Visualisiere dir die Aufteilung. Du kannst dich hierfür an dem Tutorial Notebook orientieren.
- Was stellst du fest?

In [ ]:
# Summierung der täglichen Verkäufe

daily_sales = df.groupby(df["Order Date"].dt.date)["Total Sales"].sum().reset_index()
daily_sales.columns = ["Date", "Total_Sales"]
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"])
daily_sales = daily_sales.set_index("Date").sort_index()

In [ ]:
print(f"Zeitraum: {daily_sales.index.min()} bis {daily_sales.index.max()}")
print(f"Anzahl Tage: {len(daily_sales)}")
daily_sales.head()

In [ ]:
# Führe die Zeitreihenanalyse auf einem Monatsintervallen durch
decomposition = seasonal_decompose(daily_sales["Total_Sales"], model="additive", period=30)

In [ ]:
print("Zusammenfassung der Zeitreihenkomponenten:")
print(
    f"Original-Zeitreihe: Min={daily_sales['Total_Sales'].min():.2f}, Max={daily_sales['Total_Sales'].max():.2f}"
)
print(f"Trend: Min={decomposition.trend.min():.2f}, Max={decomposition.trend.max():.2f}")
print(
    f"Saisonalität: Min={decomposition.seasonal.min():.2f}, Max={decomposition.seasonal.max():.2f}"
)
print(f"Residuen: Min={decomposition.resid.min():.2f}, Max={decomposition.resid.max():.2f}")

In [ ]:
# Visualisiere die Ergebnisse

fig, axes = plt.subplots(4, 1, figsize=(15, 12))

# Original time series
decomposition.observed.plot(ax=axes[0], title="Original Zeitreihe (Tägliche Umsätze)")
axes[0].set_ylabel("Umsatz")
axes[0].grid(True, alpha=0.3)

# Trend component
decomposition.trend.plot(ax=axes[1], title="Trend-Komponente", color="orange")
axes[1].set_ylabel("Trend")
axes[1].grid(True, alpha=0.3)

# Seasonal component
decomposition.seasonal.plot(
    ax=axes[2], title="Saisonale Komponente (30-Tage-Zyklus)", color="green"
)
axes[2].set_ylabel("Saisonalität")
axes[2].grid(True, alpha=0.3)

# Residual component
decomposition.resid.plot(ax=axes[3], title="Residuen (Rauschen)", color="red")
axes[3].set_ylabel("Residuen")
axes[3].set_xlabel("Datum")
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Fazit

Was konntest du an deinen Analysen ablesen? Was für Werbestrategien würdest du anhand diesen ableiten?

---

## **Optional: Erstellung eines Abschlussberichts**
- Formuliere deine Analyseergebnisse als zusammenhängenden Bericht.
- Struktur des Berichts:
  1. **Einleitung** – Ziele der Analyse
  2. **Datenbeschreibung** – Überblick über die Daten
  3. **Methodik** – Wie wurden die Daten bereinigt und analysiert?
  4. **Ergebnisse** – Wichtige Erkenntnisse mit Diagrammen und Kennzahlen
  5. **Fazit & Handlungsempfehlungen** – Interpretation der Ergebnisse und mögliche Maßnahmen zur Optimierung des Verkaufs.

---

